# AIC HCMC 2025 Data Download

**Before running: `Kernel → Restart & Clear Outputs`**

Downloads all data from CSV (skipping keyframes) into `data/2026/`.
Uses aria2c for parallel multi-connection downloads.

In [ ]:
import csv
import os
import subprocess
import zipfile
import shutil
from pathlib import Path

if not Path("pyproject.toml").exists() and (Path.cwd().parent / "pyproject.toml").exists():
    os.chdir(Path.cwd().parent)

SKIP_KEYFRAMES = True
ARIA_CONNECTIONS = 16
BASE_DIR = Path("data/2026")
TEMP_DIR = Path("data/downloads")

TEMP_DIR.mkdir(parents=True, exist_ok=True)

import subprocess as _sp
_ver = _sp.run(["aria2c", "--version"], capture_output=True, text=True)
print(f"aria2c {_ver.stdout.splitlines()[0]}")

In [ ]:
# Parse the CSV
rows = []
with open("aic_hcmc_data.csv") as f:
    reader = csv.DictReader(f)
    for row in reader:
        filename = row.get("Filenames", "").strip()
        url = row.get("Download link", "").strip()
        if filename and url:
            rows.append((filename, url))

print(f"Total entries: {len(rows)}")
for name, url in rows:
    print(f"  {name}")

In [ ]:
def download_via_aria2(items):
    """Download all files with aria2c. items: list of (local_name, url)."""
    urls = [url for _, url in items]
    if not urls:
        return

    for name, _ in items:
        local = TEMP_DIR / name
        aria2_ctrl = TEMP_DIR / f"{name}.aria2"
        if aria2_ctrl.exists():
            aria2_ctrl.unlink()

    cmd = [
        "aria2c",
        "-Z",
        "-d", str(TEMP_DIR),
        "-x", str(ARIA_CONNECTIONS),
        "-s", str(ARIA_CONNECTIONS),
        "-j", str(ARIA_CONNECTIONS),
        "-c",
        "--file-allocation=none",
        "--max-tries=5",
        "--retry-wait=10",
        "--console-log-level=notice",
        "--summary-interval=30",
        *urls,
    ]
    subprocess.run(cmd, check=True)
    print(f"\nAll {len(urls)} downloads complete.")

In [ ]:
def extract_zip(zip_path, target_dir, strip_prefix=True):
    """Extract a ZIP into target_dir, optionally stripping the top-level folder."""
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.namelist():
            if member.endswith("/"):
                continue
            if strip_prefix:
                parts = member.split("/", 1)
                rel_path = parts[1] if len(parts) > 1 else member
            else:
                rel_path = member
            dest = target_dir / rel_path
            dest.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(member) as src, open(dest, "wb") as dst:
                dst.write(src.read())
        print(f"  Extracted {len(zf.namelist())} entries -> {target_dir}")

In [ ]:
# Classify entries
videos = {}    # LXX -> [(filename, url), ...]
others = {}    # category -> [(filename, url), ...]

for name, url in rows:
    lower = name.lower()
    if lower.startswith("keyframes") and SKIP_KEYFRAMES:
        continue
    if lower.startswith("videos"):
        loc = name.replace(".zip", "").split("_")[1]  # L21, L26
        videos.setdefault(loc, []).append((name, url))
    elif lower.startswith("clip-features"):
        others.setdefault("clip-features-32", []).append((name, url))
    elif lower.startswith("map-keyframes"):
        others.setdefault("map-keyframes", []).append((name, url))
    elif lower.startswith("media-info"):
        others.setdefault("media-info", []).append((name, url))
    elif lower.startswith("objects"):
        others.setdefault("objects", []).append((name, url))

print("=== Videos ===")
for loc in sorted(videos.keys()):
    files = [n for n,_ in videos[loc]]
    print(f"  {loc}: {files}")
print("\n=== Other ===")
for cat in sorted(others.keys()):
    files = [n for n,_ in others[cat]]
    print(f"  {cat}: {files}")

In [ ]:
# Stage 1: Download all ZIPs (aria2c with 16 connections)
print("=" * 60)
print("STAGE 1: Downloading")
print("=" * 60)

to_download = [(n, u) for n, u in rows if not (SKIP_KEYFRAMES and n.lower().startswith("keyframes"))]
print(f"{len(to_download)} files to download via aria2c")
download_via_aria2(to_download)

In [ ]:
# Stage 2: Extract videos -> data/2026/videos/LXX/
print("=" * 60)
print("STAGE 2: Extracting videos")
print("=" * 60)

for loc in sorted(videos.keys()):
    target = BASE_DIR / "videos" / loc
    for name, url in videos[loc]:
        zip_path = TEMP_DIR / name
        if not zip_path.exists():
            print(f"  MISSING: {zip_path}")
            continue
        print(f"\n[Videos/{loc}] {name}")
        extract_zip(zip_path, target)

In [ ]:
# Stage 3: Extract other data -> data/2026/<category>/
print("=" * 60)
print("STAGE 3: Extracting metadata & features")
print("=" * 60)

CATEGORY_MAP = {
    "clip-features-32": "clip-features-32",
    "map-keyframes": "map-keyframes",
    "media-info": "media-info",
    "objects": "objects",
}

for cat, entries in others.items():
    if cat not in CATEGORY_MAP:
        print(f"  Skipping unknown: {cat}")
        continue
    target = BASE_DIR / CATEGORY_MAP[cat]
    for name, url in entries:
        zip_path = TEMP_DIR / name
        if not zip_path.exists():
            print(f"  MISSING: {zip_path}")
            continue
        print(f"\n[{cat}] {name}")
        extract_zip(zip_path, target)

In [ ]:
# Stage 4: Summary
print("=" * 60)
print("EXTRACTION SUMMARY")
print("=" * 60)

for root, dirs, files in os.walk(BASE_DIR):
    if root == str(BASE_DIR):
        continue
    level = root.replace(str(BASE_DIR) + "/", "")
    n_files = len(files)
    n_dirs = len(dirs)
    size_mb = sum((Path(root) / f).stat().st_size for f in files) // 1024 // 1024
    print(f"  {level}/  ({n_files} files, {n_dirs} dirs, ~{size_mb} MB)")

In [ ]:
# Optional: clean up temp downloads
# import shutil
# shutil.rmtree(TEMP_DIR)
# print("Temporary files cleaned.")